# AML Benchmark — Part A v2 (Account-Level Features)

**Branch:** `feature/account-level-features`

**Features:** 30 total (8 original + 18 Account-Level + 4 Derived)

**Runs:** 5 Strategien × 2 Modelle × 3 Prevalence-Levels = **30 Runs**

**Feature Caching:** Account-Level Features werden einmal berechnet.



## Schritt 1 — Google Drive mounten

In [1]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive gemountet!')

Mounted at /content/drive
Drive gemountet!


In [2]:
from pathlib import Path

backup_root = Path('/content/drive/MyDrive/aml_results')
print('Verfügbare Backups:')
for b in sorted(backup_root.iterdir()):
    runs = (b / 'runs')
    n_runs = len(list(runs.iterdir())) if runs.exists() else 0
    print(f'  {b.name} | runs={n_runs}')

Verfügbare Backups:
  large_run_v2_20260404_1637 | runs=19
  large_run_v2_20260407_1904 | runs=37
  large_run_v2_ongoing | runs=22
  part_b_multi_run_20260501_0651 | runs=0
  part_b_pai_hnu_incremental_0001_20260501_1317 | runs=0
  part_b_pai_hnu_incremental_0005_20260501_1305 | runs=0
  part_b_pai_hnu_incremental_001_20260501_1254 | runs=0
  part_b_pai_hnu_progress_20260501_124452 | runs=0
  part_b_pai_hnu_run_20260501_1322 | runs=0
  part_b_pai_hnu_smoke_20260501_1244 | runs=0
  part_b_run_20260403_1006 | runs=0
  part_b_v2_run_20260408_0634 | runs=3
  strategy6_run_20260416_2005 | runs=0
  strategy6_run_20260420_1420 | runs=0


## Schritt 2 — RAM & GPU prüfen

In [ ]:
import psutil, os
ram = psutil.virtual_memory()
print(f'Gesamt-RAM : {ram.total / 1e9:.1f} GB')
print(f'Freier RAM : {ram.available / 1e9:.1f} GB')
os.system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')
print()
print('Mindestanforderung: A100 GPU + Erweiterter RAM (179GB)')

Gesamt-RAM : 179.4 GB
Freier RAM : 176.7 GB

Mindestanforderung: A100 GPU + Erweiterter RAM (179GB)


## Schritt 3 — Projektcode klonen + Umgebung einrichten

In [7]:
import os, sys
from pathlib import Path

!git clone -b feature/account-level-features https://github.com/fdrmic/classimbalance.git /content/classimbalance

PROJECT_DIR = Path('/content/classimbalance')
os.chdir(PROJECT_DIR)

if str(PROJECT_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / 'src'))

print('Branch:', os.popen('git branch --show-current').read().strip())
print('Arbeitsverzeichnis:', os.getcwd())
print('aggregator.py:', (PROJECT_DIR / 'src/aml_benchmark/features/aggregator.py').exists())
print('feature_cache.py:', (PROJECT_DIR / 'src/aml_benchmark/features/feature_cache.py').exists())

fatal: destination path '/content/classimbalance' already exists and is not an empty directory.
Branch: feature/account-level-features
Arbeitsverzeichnis: /content/classimbalance
aggregator.py: True
feature_cache.py: True


In [5]:
!python /content/classimbalance/extract_f2_from_runs.py --summary-csv /content/drive/MyDrive/aml_results/part_a_summary_v2.csv --runs-roots /content/drive/MyDrive/aml_results/large_run_v2_ongoing/runs /content/drive/MyDrive/aml_results/large_run_v2_20260404_1637/runs /content/drive/MyDrive/aml_results/large_run_v2_20260407_1904/runs --output-csv /content/drive/MyDrive/aml_results/part_a_summary_v2_with_f2.csv

python3: can't open file '/content/classimbalance/extract_f2_from_runs.py': [Errno 2] No such file or directory


In [6]:
!ls /content/classimbalance/extract_f2_from_runs.py
!ls /content/classimbalance/*.py

ls: cannot access '/content/classimbalance/extract_f2_from_runs.py': No such file or directory
/content/classimbalance/test_pipeline.py


## Schritt 4 — Dependencies installieren

In [ ]:
import os, sys
from pathlib import Path

PROJECT_DIR = Path('/content/classimbalance')
os.chdir(PROJECT_DIR)

if str(PROJECT_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / 'src'))

!pip install -e . -q
!pip install -r requirements.txt -q

print('Installation abgeschlossen!')

from aml_benchmark.features.aggregator import ACCOUNT_FEATURE_NAMES
from aml_benchmark.features.pipeline import FEATURE_NAMES
from aml_benchmark.features.feature_cache import cache_exists
print(f'Account Features : {len(ACCOUNT_FEATURE_NAMES)}')
print(f'Total Features   : {len(FEATURE_NAMES)} (vorher: 8)')
print(f'Caching Module   : OK')

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 150.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.5/77.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 94.9 MB/s eta 0:00:00
  Building editable for aml_benchmark (pyproject.toml) ... done
Installation abgeschlossen!
Account Features : 18
Total Features   : 30 (vorher: 8)
Caching Module   : OK


In [ ]:
!git pull

Already up to date.


## Schritt 5 — Large Files auf Drive prüfen

In [ ]:
from pathlib import Path

DRIVE_DIR = Path('/content/drive/MyDrive/aml_data')

for fname in ['LI-Large_Trans.csv', 'LI-Large_accounts.csv', 'LI-Large_Patterns.txt']:
    p = DRIVE_DIR / fname
    if p.exists():
        print(f'  OK    {fname}  ({p.stat().st_size / 1e9:.2f} GB)')
    else:
        print(f'  FEHLT {fname}')

  OK    LI-Large_Trans.csv  (16.74 GB)
  OK    LI-Large_accounts.csv  (0.14 GB)
  OK    LI-Large_Patterns.txt  (0.00 GB)


## Schritt 6 — paths_large_v2.yaml für Colab erstellen

In [ ]:
import yaml, os
from pathlib import Path

PROJECT_DIR = Path(os.getcwd())
DRIVE_DIR   = Path('/content/drive/MyDrive/aml_data')

with open(PROJECT_DIR / 'configs' / 'paths_large_v2.yaml') as f:
    cfg = yaml.safe_load(f)

cfg['raw_dir']         = str(DRIVE_DIR)
cfg['processed_dir']   = str(PROJECT_DIR / 'data' / 'processed_v2')
cfg['splits_dir']      = str(PROJECT_DIR / 'data' / 'splits_v2')
cfg['outputs_dir']     = str(PROJECT_DIR / 'outputs' / 'runs_v2')
cfg['leaderboard_dir'] = str(PROJECT_DIR / 'outputs' / 'leaderboard_v2')

with open(PROJECT_DIR / 'configs' / 'paths_large_v2.yaml', 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('paths_large_v2.yaml aktualisiert:')
print(yaml.dump(cfg, default_flow_style=False))

paths_large_v2.yaml aktualisiert:
accounts_filename: LI-Large_accounts.csv
leaderboard_dir: /content/classimbalance/outputs/leaderboard_v2
output_transactions_labeled: transactions_labeled.parquet
outputs_dir: /content/classimbalance/outputs/runs_v2
part_a_summary: part_a_summary_v2.csv
patterns_filename: LI-Large_Patterns.txt
processed_dir: /content/classimbalance/data/processed_v2
raw_dir: /content/drive/MyDrive/aml_data
split_manifest: split_manifest.json
splits_dir: /content/classimbalance/data/splits_v2
transactions_filename: LI-Large_Trans.csv



## Schritt 7 — Verfügbare Backups prüfen
Prüft ob processed/splits bereits auf Drive vorhanden sind.

In [ ]:
from pathlib import Path

backup_root = Path('/content/drive/MyDrive/aml_results')
print('Verfügbare Backups:')
for b in sorted(backup_root.iterdir()):
    runs = b / 'runs'
    n_runs = len(list(runs.iterdir())) if runs.exists() else 0
    print(f'  {b.name} | runs={n_runs}')

Verfügbare Backups:
  large_run_20260331_2107 | runs=0
  large_run_20260401_0756 | runs=0
  large_run_20260401_1440 | runs=30
  large_run_20260402_0656 | runs=30
  large_run_v2_20260404_1237 | runs=2
  large_run_v2_20260404_1637 | runs=19
  large_run_v2_ongoing | runs=18
  part_b_run_20260403_1006 | runs=0


In [ ]:
# Splits + Processed von Drive laden
import shutil, os
from pathlib import Path

PROJECT_DIR = Path(os.getcwd())
backup = Path('/content/drive/MyDrive/aml_results/large_run_v2_20260404_1637')

for folder, dst_path in [
    ('processed', PROJECT_DIR / 'data' / 'processed_v2'),
    ('splits',    PROJECT_DIR / 'data' / 'splits_v2'),
    ('runs',      PROJECT_DIR / 'outputs' / 'runs_v2'),
]:
    src = backup / folder
    if src.exists():
        dst_path.mkdir(parents=True, exist_ok=True)
        shutil.copytree(src, dst_path, dirs_exist_ok=True)
        print(f'Geladen: {folder}/')
    else:
        print(f'Nicht vorhanden: {folder}/ — wird neu erstellt')

# Ongoing-Backup dazuladen (alle neuen runs)
ongoing = Path('/content/drive/MyDrive/aml_results/large_run_v2_ongoing')
runs_src = ongoing / 'runs'
runs_dst = PROJECT_DIR / 'outputs' / 'runs_v2'
if runs_src.exists():
    runs_dst.mkdir(parents=True, exist_ok=True)
    shutil.copytree(runs_src, runs_dst, dirs_exist_ok=True)
    print(f'Ongoing runs dazugeladen.')
else:
    print('Keine ongoing runs gefunden.')

# Splits prüfen
splits_dir = PROJECT_DIR / 'data' / 'splits_v2'
for f in ['train.parquet', 'val.parquet', 'test.parquet']:
    p = splits_dir / f
    status = f'{p.stat().st_size / 1e9:.2f} GB' if p.exists() else 'FEHLT'
    print(f'  {f}: {status}')

# Runs prüfen
runs_dir = PROJECT_DIR / 'outputs' / 'runs_v2'
if runs_dir.exists():
    valid_runs = [
        r for r in runs_dir.iterdir()
        if r.is_dir()
        and (r / 'metrics_test.json').exists()
        and (r / 'run_config.json').exists()
    ]
    print(f'Valide Runs geladen: {len(valid_runs)}')
    for r in sorted(valid_runs):
        print(f'  {r.name}')
else:
    print('Kein runs/ Ordner gefunden')

Geladen: processed/
Geladen: splits/
Geladen: runs/
Ongoing runs dazugeladen.
  train.parquet: 3.86 GB
  val.parquet: 0.82 GB
  test.parquet: 0.83 GB
Valide Runs geladen: 30
  random_forest__adasyn__p001__20260407_124424
  random_forest__adasyn__p005__20260407_122508
  random_forest__adasyn__p010__20260407_120621
  random_forest__baseline__p001__20260404_140031
  random_forest__baseline__p005__20260404_134339
  random_forest__baseline__p010__20260404_132609
  random_forest__class_weighting__p001__20260407_153534
  random_forest__class_weighting__p005__20260407_151823
  random_forest__class_weighting__p010__20260407_132552
  random_forest__random_undersampling__p001__20260404_145300
  random_forest__random_undersampling__p005__20260404_144450
  random_forest__random_undersampling__p010__20260404_143732
  random_forest__smote__p001__20260407_081802
  random_forest__smote__p005__20260407_071352
  random_forest__smote__p010__20260407_065056
  xgboost__adasyn__p001__20260407_131823
  xgboos

In [ ]:
ongoing = Path('/content/drive/MyDrive/aml_results/large_run_v2_ongoing')
for item in sorted(ongoing.iterdir()):
    print(item.name)
    if item.is_dir():
        for sub in sorted(item.iterdir()):
            print(f'  {sub.name}')

runs
  random_forest__adasyn__p001__20260407_124424
  random_forest__adasyn__p005__20260407_122508
  random_forest__adasyn__p010__20260407_120621
  random_forest__class_weighting__p001__20260407_153534
  random_forest__class_weighting__p005__20260407_151823
  random_forest__class_weighting__p010__20260407_132552
  random_forest__smote__p001__20260407_081802
  random_forest__smote__p005__20260407_071352
  random_forest__smote__p010__20260407_065056
  xgboost__adasyn__p001__20260407_131823
  xgboost__adasyn__p005__20260407_131054
  xgboost__adasyn__p010__20260407_130303
  xgboost__class_weighting__p001__20260407_160620
  xgboost__class_weighting__p005__20260407_155941
  xgboost__class_weighting__p010__20260407_155304
  xgboost__smote__p001__20260407_085048
  xgboost__smote__p005__20260407_084334
  xgboost__smote__p010__20260407_083543
splits
  feature_pipeline_v2.pkl
  split_manifest.json
  test.parquet
  test_features_v2.parquet
  train.parquet
  train_features_v2.parquet
  val.parquet


## Schritt 8 — Daten einlesen & labeln
**Nur ausführen falls kein Backup geladen werden konnte!**

⚠️ Dauert ~20 Minuten

In [ ]:
# Nur ausführen falls processed_v2 leer ist
from pathlib import Path
p = Path('/content/classimbalance/data/processed_v2/transactions_labeled.parquet')
if p.exists():
    print(f'Bereits vorhanden ({p.stat().st_size / 1e9:.2f} GB) — Schritt 8 überspringen!')
else:
    print('Nicht vorhanden — erstelle neu ...')
    import os
    os.system('python -m aml_benchmark.data.make_dataset --paths configs/paths_large_v2.yaml')

## Schritt 9 — Zeitliche Splits erstellen
**Nur ausführen falls kein Backup geladen werden konnte!**

⚠️ Dauert ~10 Minuten

In [ ]:
from pathlib import Path
p = Path('/content/classimbalance/data/splits_v2/train.parquet')
if p.exists():
    print(f'Splits bereits vorhanden — Schritt 9 überspringen!')
else:
    print('Splits fehlen — erstelle neu ...')
    import os
    os.system('python -m aml_benchmark.data.splitter --paths configs/paths_large_v2.yaml')

Splits bereits vorhanden — Schritt 9 überspringen!


In [ ]:
import os, sys
from pathlib import Path
PROJECT_DIR = Path('/content/classimbalance')
os.chdir(PROJECT_DIR)
if str(PROJECT_DIR / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR / 'src'))
!git pull
!pip install -e . -q
print('Done!')

Already up to date.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for aml_benchmark (pyproject.toml) ... done
Done!


## Schritt 10 — Anti-Timeout Javascript
⚠️ **Vor Schritt 11 ausführen** — verhindert Colab-Timeout während Feature Engineering!

In [ ]:
%%javascript
function ClickConnect(){
    console.log("Keeping alive...");
    document.querySelector("colab-toolbar-button#connect").click()
}
setInterval(ClickConnect, 60000)

<IPython.core.display.Javascript object>

In [ ]:
import psutil
ram = psutil.virtual_memory()
print(f"Total: {ram.total/1e9:.1f} GB")
print(f"Available: {ram.available/1e9:.1f} GB")
print(f"Used: {ram.used/1e9:.1f} GB")

Total: 179.4 GB
Available: 175.9 GB
Used: 1.8 GB


## Schritt 11 — 30-Run Benchmark Part A v2
**5 Strategien × 2 Modelle × 3 Prevalence-Levels = 30 Runs**

**Run 1:** Feature Engineering (~5h) + Training (~10min) — Features werden gecacht

**Runs 2-30:** Cache laden (~30s) + Training (~10min) = ~5h total für alle 29 restlichen Runs

⚠️ Über Nacht laufen lassen!

In [ ]:
!python -m aml_benchmark.experiments.grid_runner \
    --paths configs/paths_large_v2.yaml

2026-04-07 15:18:23 | INFO     | __main__ | ==============================================================
2026-04-07 15:18:23 | INFO     | __main__ | PART A BENCHMARK GRID
2026-04-07 15:18:23 | INFO     | __main__ |   Models      : ['random_forest', 'xgboost']
2026-04-07 15:18:23 | INFO     | __main__ |   Strategies  : ['baseline', 'random_undersampling', 'smote', 'adasyn', 'class_weighting']
2026-04-07 15:18:23 | INFO     | __main__ |   Prevalences : ['1.000%', '0.500%', '0.100%']
2026-04-07 15:18:23 | INFO     | __main__ |   Total runs  : 30
2026-04-07 15:18:23 | INFO     | __main__ | ==============================================================
2026-04-07 15:18:23 | INFO     | __main__ | [1/30] model=random_forest  strategy=baseline  prevalence=1.000%
2026-04-07 15:18:23 | INFO     | __main__ |   SKIPPING -- completed run found: random_forest__baseline__p010__20260404_132609
2026-04-07 15:18:23 | INFO     | __main__ | [2/30] model=random_forest  strategy=baseline  prevalence=0.500

## Schritt 12 — Threshold-Optimierung

In [ ]:
!python -m aml_benchmark.experiments.re_evaluate --paths configs/paths_large_v2.yaml

2026-04-07 18:13:40 | INFO     | __main__ | Loading val and test splits once ...
2026-04-07 18:13:59 | INFO     | aml_benchmark.utils.io | Loaded 26,409,984 rows <- /content/classimbalance/data/splits_v2/val.parquet
2026-04-07 18:14:18 | INFO     | aml_benchmark.utils.io | Loaded 26,409,984 rows <- /content/classimbalance/data/splits_v2/test.parquet
2026-04-07 18:14:18 | INFO     | __main__ | Found 37 run folders to check.
2026-04-07 18:14:18 | INFO     | __main__ | Re-evaluating random_forest__adasyn__p001__20260407_124424 ...
2026-04-07 18:14:19 | INFO     | aml_benchmark.features.feature_cache | Loading feature cache <- /content/classimbalance/data/splits_v2/val_features_v2.parquet
2026-04-07 18:14:21 | INFO     | aml_benchmark.features.feature_cache | Loading feature cache <- /content/classimbalance/data/splits_v2/test_features_v2.parquet
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  33 tasks      | elapsed:   12.1s
[Par

## Schritt 13 — Leaderboard aggregieren

In [ ]:
!python -m aml_benchmark.experiments.aggregate --paths configs/paths_large_v2.yaml

2026-04-07 19:04:31 | INFO     | __main__ | Aggregated 30 Part A runs.
2026-04-07 19:04:31 | INFO     | __main__ | Leaderboard saved -> /content/classimbalance/outputs/leaderboard_v2/part_a_summary_v2.csv  (30 rows)

  PART A LEADERBOARD  (top 10 of 30 runs)
  Showing: optimal-threshold metrics
  sorted by pr_auc_test DESC, recall DESC
     model              strategy  target_prevalence  pr_auc_test  optimal_threshold  recall_test_thresh  precision_test_thresh  f1_test_thresh  tp_test_thresh  fp_test_thresh  fn_test_thresh
0  xgboost              baseline           0.001000     0.107854           0.040491            0.239256               0.090925        0.131772     4710.000000    47091.000000    14976.000000
1  xgboost              baseline           0.005000     0.107854           0.040491            0.239256               0.090925        0.131772     4710.000000    47091.000000    14976.000000
2  xgboost              baseline           0.010000     0.107854           0.040491      

## Schritt 14 — Ergebnisse auf Drive sichern
⚠️ Immer ausführen bevor die Session endet!

In [ ]:
import shutil, datetime, os
from pathlib import Path

PROJECT_DIR = Path(os.getcwd())
ts = datetime.datetime.now().strftime('%Y%m%d_%H%M')
backup_dir = Path(f'/content/drive/MyDrive/aml_results/large_run_v2_{ts}')
backup_dir.mkdir(parents=True, exist_ok=True)

to_backup = {
    'outputs/runs_v2':        'runs',
    'outputs/leaderboard_v2': 'leaderboard',
    'data/processed_v2':      'processed',
    'data/splits_v2':         'splits',
}

for src_rel, dst_name in to_backup.items():
    src = PROJECT_DIR / src_rel
    if src.exists():
        shutil.copytree(src, backup_dir / dst_name, dirs_exist_ok=True)
        print(f'Gesichert: {src_rel}/')
    else:
        print(f'Nicht gefunden (übersprungen): {src_rel}/')

print(f'\nBackup abgeschlossen: {backup_dir}')
print(f'part_a_summary_v2.csv: {backup_dir}/leaderboard/part_a_summary_v2.csv')

Gesichert: outputs/runs_v2/
Gesichert: outputs/leaderboard_v2/
Gesichert: data/processed_v2/
Gesichert: data/splits_v2/

Backup abgeschlossen: /content/drive/MyDrive/aml_results/large_run_v2_20260407_1904
part_a_summary_v2.csv: /content/drive/MyDrive/aml_results/large_run_v2_20260407_1904/leaderboard/part_a_summary_v2.csv


---
## Hinweise

**Feature Caching — wie es funktioniert:**
Run 1 berechnet Account-Level Features und speichert sie in `data/splits_v2/`:
- `train_features_v2.parquet` (~30GB)
- `val_features_v2.parquet` (~7GB)
- `test_features_v2.parquet` (~7GB)
- `feature_pipeline_v2.pkl`

Runs 2-30 laden diese Dateien direkt — kein Re-Computing.

**Falls Session abbricht nach Run 1:**
Schritt 14 sichert alles auf Drive inkl. Feature Cache. Beim Neustart Schritt 7 verwenden und neuesten `large_run_v2_*` Backup laden — Cache ist dann sofort verfügbar.

**Session beenden:**
Nach Schritt 14: Laufzeit → Sitzung trennen und löschen.